<a href="https://colab.research.google.com/github/olympiasau/ds2002-fa26/blob/main/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(total_revenue)
print(total_units)

explanation1 = 'The 400 rows (orders) made $8,520 in revenue--falling between the estimated 8k-9k--from the 783 units sold.'
print(explanation1)
df.head()

8520.0
783
The 400 rows (orders) made $8,520 in revenue--falling between the estimated 8k-9k--from the 783 units sold.


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = (
    df.groupby('category')[['revenue']]
    .sum()
    .sort_values('revenue', ascending=False))

by_category['share_pct'] = (by_category['revenue'] / total_revenue * 100)
print(by_category)

explanation2 = 'The dataframe shows which category brings in the most income by unit amount and proportion, represented as a percent.'
print(explanation2)

          revenue  share_pct
category                    
Food       4293.0  50.387324
Merch      1771.5  20.792254
Drink      1554.0  18.239437
RainGear    901.5  10.580986
The dataframe shows which category brings in the most income by unit amount and proportion, represented as a percent.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_avg = (
    df.groupby('vendor_id')['revenue']
    .agg(avg_order_revenue = 'mean', orders='count')
    .sort_values('avg_order_revenue', ascending=False))

explanation3 = 'After grouping by the vendors and then finding each ones average, it can be seen that V-01, V-18, and V-05 have hte highest average order revenue.'
print(explanation3)
vendor_avg

After grouping by the vendors and then finding each ones average, it can be seen that V-01, V-18, and V-05 have hte highest average order revenue.


,avg_order_revenue,orders
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch = df[df['category'] == 'Merch']
share = 100 * merch['revenue'].sum()/df['revenue'].sum()

explanation4 = 'Merch makes up roughly a fifth of total revenue (exactly the percentage below).'
print(explanation4)
print(share)

Merch makes up roughly a fifth of total revenue (exactly the percentage below).
20.79225352112676


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on = 'vendor_id', how = 'left', validate='many_to_one', indicator = True)

assert len(joined) == len(df)
assert joined ['revenue'].sum() == df['revenue'].sum()

missing_rows = joined[joined['_merge'] == 'left_only']
print(missing_rows['vendor_id'].unique(), len(missing_rows), missing_rows['revenue'].sum())

joined['vendor_name'] = joined['vendor_name'].fillna('unknown vendor')

['V-18'] 108 2349.0


The unmatched vendor was V-18, which had 108 orders with a revenue of 2,349 units. While it could be simply removed from the table this would hide the significant portion of revenue generated by said vendor. Instead, I chose to label the vendor as unknown to reflect the problem in the data.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_table = joined.pivot_table(index='vendor_name', columns='category', values='revenue', aggfunc='sum', margins=True)
pivot_table

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
unknown vendor,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) For next game I would recommend that Hoos Burgers try to sell more drinks. While food makes up over 50% of revenue, drinks are only bringing in ~18% of total revenue. Hoos Burgers in particular has the most food revenue of any vendor (1,338), but the least drink revenue (171). Assuming drinks are traditionally purchased with food, this may be a good opportunity to increase sales. For example, providing incentive to add a drink for a discounted price may persuade customers who were not otherwise considering a drink, but were certain they wanted food.  

b) The least trustworthy answer is Question 6, because its biggest row is an unknown vendor. V-18 had 2,349 in revenue, making up over a quarter of the total revenue (27.6%), but it is difficult to identify which vendors are the most successful when the top seller is unnamed.